In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import pandas as pd
from cycler import cycler

warnings.filterwarnings('ignore')

INPUT_CANDIDATES = [
    Path('RobustezzaNoIsolati.xlsx'),
    Path(r'C:\Moltbok\RobustezzaNoIsolati.xlsx'),
]
INPUT_FILE = next((path for path in INPUT_CANDIDATES if path.exists()), INPUT_CANDIDATES[0])
OUTPUT_DIR = Path('figure_robustness')

PALETTE = {
    'primary': '#1F3A5F',
    'secondary': '#6E8FAF',
    'teal': '#1B7F8C',
    'orange': '#C97B4A',
    'support': '#4A5568',
    'event': '#8F3B2F',
}

SERIES_COLORS = {
    'random': PALETTE['primary'],
    'targeted_indegree': PALETTE['orange'],
    'targeted_outdegree': PALETTE['teal'],
    'gap_indegree': PALETTE['orange'],
    'gap_outdegree': PALETTE['teal'],
}

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 35
plt.rcParams['axes.labelsize'] = 17
plt.rcParams['xtick.labelsize'] = 13
plt.rcParams['ytick.labelsize'] = 13
plt.rcParams['legend.fontsize'] = 13
plt.rcParams['legend.frameon'] = False
plt.rcParams['lines.linewidth'] = 2.5
plt.rcParams['axes.linewidth'] = 0.8
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['axes.grid'] = True
plt.rcParams['axes.grid.axis'] = 'x'
plt.rcParams['grid.alpha'] = 0.2
plt.rcParams['grid.linestyle'] = '--'
plt.rcParams['grid.linewidth'] = 0.6
plt.rcParams['xtick.major.size'] = 4
plt.rcParams['ytick.major.size'] = 4
plt.rcParams['xtick.direction'] = 'out'
plt.rcParams['ytick.direction'] = 'out'
plt.rcParams['axes.prop_cycle'] = cycler(color=[PALETTE['primary'], PALETTE['secondary'], PALETTE['teal'], PALETTE['orange']])

In [ ]:
if not INPUT_FILE.exists():
    raise FileNotFoundError(f'Input file not found: {INPUT_FILE.resolve()}')

robustness_daily = pd.read_excel(INPUT_FILE, sheet_name='robustness_daily')
robustness_daily['day'] = pd.to_datetime(robustness_daily['day'], errors='coerce')
robustness_daily['percent_removed'] = pd.to_numeric(robustness_daily['percent_removed'], errors='coerce')
robustness_daily['gc_fraction'] = pd.to_numeric(robustness_daily['gc_fraction'], errors='coerce')
robustness_daily = (
    robustness_daily
    .dropna(subset=['day', 'percent_removed', 'gc_fraction', 'attack'])
    .sort_values(['day', 'percent_removed', 'attack'])
    .reset_index(drop=True)
 )

print(f'Loaded data: {robustness_daily.shape[0]:,} rows')
print(f'Date range: {robustness_daily["day"].min().date()} -> {robustness_daily["day"].max().date()}')
print(f'Percentages: {sorted(robustness_daily["percent_removed"].unique())}')

Dati caricati: 1,272 righe
Intervallo: 2026-01-28 -> 2026-03-21
Percentuali: [np.float64(0.0), np.float64(0.01), np.float64(0.05), np.float64(0.1), np.float64(0.2), np.float64(0.4), np.float64(0.6), np.float64(0.8)]


In [ ]:
KEY_DATES = pd.to_datetime([
    '2026-01-30', '2026-01-31',
    '2026-02-05', '2026-02-06', '2026-02-07', '2026-02-08', '2026-02-09'
])

def apply_academic_style(ax, remove_top_right=True):
    if remove_top_right:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.8)
    ax.spines['bottom'].set_linewidth(0.8)
    ax.spines['left'].set_color('#333333')
    ax.spines['bottom'].set_color('#333333')
    ax.tick_params(colors='#222222', width=0.8, length=4)
    ax.grid(False)
    ax.grid(True, axis='x', alpha=0.2, linestyle='--', linewidth=0.6, color='#8A8A8A')
    ax.set_axisbelow(True)
    for line in ax.get_lines():
        if line.get_alpha() is not None and line.get_alpha() < 0.5:
            continue
        if line.get_linestyle() in ('-', 'solid'):
            line.set_linewidth(max(line.get_linewidth(), 2.8))
            if line.get_marker() not in [None, 'None', '', ' ']:
                line.set_markersize(max(line.get_markersize(), 3))

def add_key_dates_vlines(ax, dates=None, color=None, alpha=0.15, linewidth=1.0, data_range=None):
    dates = KEY_DATES if dates is None else dates
    color = PALETTE['event'] if color is None else color
    if data_range is not None:
        dates = [date for date in dates if data_range[0] <= date <= data_range[1]]
    highlight_date = pd.Timestamp('2026-02-09')
    for date in dates:
        if pd.Timestamp(date).normalize() == highlight_date:
            ax.axvline(date, color='#B5122B', alpha=max(alpha, 0.9), linewidth=max(linewidth, 2.3), linestyle='--', zorder=0)
        else:
            ax.axvline(date, color=color, alpha=alpha, linewidth=linewidth, linestyle='--', zorder=0)

def save_figure(fig, base_name):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    png_path = OUTPUT_DIR / f'{base_name}.png'
    pdf_path = OUTPUT_DIR / f'{base_name}.pdf'
    fig.savefig(png_path, dpi=300, bbox_inches='tight')
    fig.savefig(pdf_path, dpi=300, bbox_inches='tight')
    print(f'Saved: {png_path} and {pdf_path}')

In [ ]:
# === FIGURE: Robustness curve S(q) / S(0) ===
# The curve represents the last cumulative network available in the input file.
# Thresholds missing from the Excel file remain NaN: they are not interpolated.
required_q = [0.00, 0.01, 0.05, 0.10, 0.20, 0.40, 0.60, 0.80]
last_day = robustness_daily['day'].max()
last_day_data = robustness_daily[robustness_daily['day'] == last_day].copy()

q_curve = pd.DataFrame({'q': required_q})
q_curve['percent_removed'] = q_curve['q']

fig, ax = plt.subplots(figsize=(9, 5))
label_map = {
    'random': 'Random removal',
    'targeted_indegree': 'Targeted in-degree',
    'targeted_outdegree': 'Targeted out-degree',
}

for attack_type in ['random', 'targeted_indegree', 'targeted_outdegree']:
    attack_data = (
        last_day_data[last_day_data['attack'] == attack_type]
        .groupby('percent_removed', as_index=False)['gc_fraction']
        .mean()
        .rename(columns={'gc_fraction': 'normalized_giant_wcc'})
    )
    curve = q_curve[['q', 'percent_removed']].merge(attack_data, on='percent_removed', how='left')
    curve.loc[curve['q'] == 0, 'normalized_giant_wcc'] = 1.0
    curve = curve.sort_values('q')
    ax.plot(
        curve['q'] * 100,
        curve['normalized_giant_wcc'],
        color=SERIES_COLORS[attack_type],
        linewidth=2.2,
        marker='o',
        markersize=4,
        label=label_map[attack_type],
    )

missing_q = sorted(set(required_q) - set(last_day_data['percent_removed'].unique()))
if missing_q:
    print(
        f'Warning: for {last_day.date()}, the following thresholds are missing from the file: '
        f'{[int(q * 100) for q in missing_q if q > 0]}%; the corresponding points remain absent.'
    )

apply_academic_style(ax)
ax.set_xlabel('Percentage of non-isolated nodes removed, $q$ (%)')
ax.set_ylabel(r'Normalized giant WCC size, $S(q)/S(0)$')
ax.set_xlim(0, 80)
ax.set_ylim(0, 1.05)
ax.set_xticks([q * 100 for q in required_q])
ax.set_xticklabels(['0', '1', '5', '10', '20', '40', '60', '80'])
ax.legend(loc='best', frameon=False)
plt.tight_layout()
save_figure(fig, 'fig_robustness_curve_normalized')
plt.close(fig)

Salvate: figure_robustness\fig_robustness_curve_normalized.png e figure_robustness\fig_robustness_curve_normalized.pdf


In [ ]:
# === FIGURE: Corrected robustness curve S(q) / S(0) ===
# S(q) is the absolute giant WCC size after removal.
# The denominator remains the absolute giant WCC size before removal.
dinamica_file = Path('DinamicaFinale1.xlsx')
if not dinamica_file.exists():
    dinamica_file = Path(r'C:\Moltbok\DinamicaFinale1.xlsx')

if not dinamica_file.exists():
    raise FileNotFoundError(
        f'DinamicaFinale1 file not found: {dinamica_file.resolve()}. '
        'The nodes, isolated_nodes, and giant_wcc_pct columns are required.'
    )

daily_metrics = pd.read_excel(dinamica_file, sheet_name='daily_all_metrics')
daily_metrics['day'] = pd.to_datetime(daily_metrics['day'], errors='coerce')

required_baseline_columns = {'day', 'nodes', 'isolated_nodes', 'giant_wcc_pct'}
missing_baseline_columns = required_baseline_columns - set(daily_metrics.columns)
if missing_baseline_columns:
    raise ValueError(
        f'The file {dinamica_file} is missing the required columns: '
        f'{sorted(missing_baseline_columns)}'
    )

baseline_rows = daily_metrics[daily_metrics['day'] == last_day]
if baseline_rows.empty:
    raise ValueError(
        f'The file {dinamica_file} has no baseline for the date {last_day.date()}.'
    )
baseline = baseline_rows.iloc[0]
non_isolated_nodes = float(baseline['nodes']) - float(baseline['isolated_nodes'])
s0 = float(baseline['giant_wcc_pct']) / 100.0 * float(baseline['nodes'])
if non_isolated_nodes <= 0 or s0 <= 0:
    raise ValueError('Unable to calculate S(q)/S(0): invalid baseline.')

q_curve = pd.DataFrame({'q': required_q})
q_curve['percent_removed'] = q_curve['q']

fig, ax = plt.subplots(figsize=(9, 5))
label_map = {
    'random': 'Random removal',
    'targeted_indegree': 'Targeted in-degree',
    'targeted_outdegree': 'Targeted out-degree',
}

for attack_type in ['random', 'targeted_indegree', 'targeted_outdegree']:
    attack_data = last_day_data[last_day_data['attack'] == attack_type].copy()
    attack_data['remaining_non_isolated'] = non_isolated_nodes - attack_data['removed_nodes']
    attack_data['giant_wcc_size'] = attack_data['gc_fraction'] * attack_data['remaining_non_isolated']
    attack_data['normalized_giant_wcc'] = attack_data['giant_wcc_size'] / s0
    attack_data = (
        attack_data
        .groupby('percent_removed', as_index=False)['normalized_giant_wcc']
        .mean()
    )
    curve = q_curve[['q', 'percent_removed']].merge(attack_data, on='percent_removed', how='left')
    curve.loc[curve['q'] == 0, 'normalized_giant_wcc'] = 1.0
    curve = curve.sort_values('q')
    ax.plot(
        curve['q'] * 100,
        curve['normalized_giant_wcc'],
        color=SERIES_COLORS[attack_type],
        linewidth=2.2,
        marker='o',
        markersize=4,
        label=label_map[attack_type],
    )

missing_q = sorted(set(required_q) - set(last_day_data['percent_removed'].unique()))
if missing_q:
    print(
        f'Warning: for {last_day.date()}, the following thresholds are missing from the file: '
        f'{[int(q * 100) for q in missing_q if q > 0]}%; the corresponding points remain absent.'
    )

print(f'Absolute baseline loaded from {dinamica_file}: S(0) = {s0:.0f} nodes')
apply_academic_style(ax)
ax.set_xlabel('Percentage of non-isolated nodes removed, $q$ (%)')
ax.set_ylabel(r'Normalized giant WCC size, $S(q)/S(0)$')
ax.set_xlim(0, 80)
ax.set_ylim(0, 1.05)
ax.set_xticks([q * 100 for q in required_q])
ax.set_xticklabels(['0', '1', '5', '10', '20', '40', '60', '80'])
ax.legend(loc='best', frameon=False)
plt.tight_layout()
save_figure(fig, 'fig_robustness_curve_normalized_corrected')
plt.close(fig)


Baseline assoluta caricata da DinamicaFinale1.xlsx: S(0) = 63107 nodi
Salvate: figure_robustness\fig_robustness_curve_normalized_corrected.png e figure_robustness\fig_robustness_curve_normalized_corrected.pdf
